# เทรน PCB YOLO11 บน Colab และเก็บ progress ใน Google Drive

เลือก **Runtime → Change runtime type → GPU** แล้วรัน cell ตามลำดับ

- ครั้งแรกเริ่มจาก YOLO11n pretrained
- บันทึกโมเดลและ log ลง Drive ทุก epoch พร้อม snapshot สองรุ่นเพื่อใช้ resume
- เมื่อ quota หมดหรือ runtime ถูกลบ ให้รอ GPU กลับมา แล้วรัน Notebook ตั้งแต่ต้นด้วย Drive folder เดิมและ `NEW_RUN = False`
- งานใน epoch ที่ยังไม่บันทึกต้องทำใหม่ ไม่สามารถกู้ state กลาง batch หรือเริ่ม runtime ใหม่อัตโนมัติเมื่อ quota ยังไม่พร้อม
- ถ้าจบรอบแล้ว จะไม่เริ่มเทรนซ้ำ ใช้ `NEW_RUN = True` เฉพาะเมื่อต้องการทดลองรอบใหม่

Checkpoint ถูกเขียนลง Drive ระหว่างฝึก จึงไม่ได้รอคัดลอกหลังเทรนจบ อย่างไรก็ดี การ sync Drive อาจล่าช้าหรือผิดพลาดได้ ไม่รับรองการกู้คืนเมื่อไฟล์สำรองทุกชุดเสียหาย


In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('เลือก GPU runtime ก่อน หรือรอให้โควตา GPU พร้อมใช้งาน')
print('GPU:', torch.cuda.get_device_name(0))


## รับโค้ด ZIP และ pretrained weights จาก GitHub

Cell นี้ใช้ Git LFS เพื่อรับไฟล์จริง โค้ดและ dataset อยู่ใน `/content` ซึ่งต้องเตรียมใหม่เมื่อ runtime ถูกลบ


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO = Path('/content/Yolo11_PCB_pre-operator')
if shutil.which('git-lfs') is None:
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'git-lfs'], check=True)
if not (REPO / '.git').is_dir():
    env = dict(os.environ, GIT_LFS_SKIP_SMUDGE='1')
    subprocess.run(['git', 'clone', 'https://github.com/Jadsadar/Yolo11_PCB_pre-operator.git', str(REPO)],
                   check=True, env=env)
os.chdir(REPO)
subprocess.run(['git', 'lfs', 'install', '--local'], check=True)
subprocess.run(['git', 'pull', '--ff-only', 'origin', 'main'], check=True)
subprocess.run(['git', 'lfs', 'pull'], check=True)
print('Project:', REPO)


## ติดตั้ง dependency สำหรับ Colab และแตกข้อมูล

ใช้ `requirements-colab.txt` เพื่อคง PyTorch/CUDA ที่ Colab เตรียมไว้ ไม่ติดตั้งส่วน UI เดสก์ท็อป


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-colab.txt'], check=True)
subprocess.run([sys.executable, '-m', 'scripts.download_pretrained'], check=True)

from zipfile import ZipFile
from pathlib import PurePosixPath
with ZipFile(REPO / 'PCB Defect.v1-yolov5.yolov11.zip') as archive:
    for member in archive.infolist():
        parts = PurePosixPath(member.filename).parts
        if not parts or parts[0] not in {'train', 'valid', 'test'}:
            continue
        if '..' in parts or not (REPO / member.filename).resolve().is_relative_to(REPO.resolve()):
            raise ValueError('Unsafe dataset archive path')
        archive.extract(member, REPO)
print('Dataset extracted; repository data.yaml is preserved.')


## เชื่อม Google Drive และตั้งค่าการทดลอง

อนุญาตให้ Colab mount Drive ตามหน้าจอ ใช้โฟลเดอร์เดิมทุกครั้งที่ต้องการ resume และอย่ารันสอง runtime เขียน experiment เดียวกันพร้อมกัน


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = Path('/content/drive/MyDrive/PCB_YOLO11/experiments')
if not Path('/content/drive/MyDrive').is_dir():
    raise RuntimeError('Google Drive is not mounted')
PROJECT.mkdir(parents=True, exist_ok=True)

import yaml
config_path = REPO / 'configs/train.yaml'
settings = yaml.safe_load(config_path.read_text(encoding='utf-8'))
for key in ('data', 'model'):
    settings[key] = str((config_path.parent / settings[key]).resolve())
settings.update(project=str(PROJECT), device=0, amp=True, workers=2, batch=8, tensorboard=True)
# ตั้ง epochs สำหรับการทดลองใหม่ การ resume ใช้เป้าหมาย epochs จาก checkpoint เดิม
settings['epochs'] = 100
COLAB_CONFIG = REPO / 'tmp/colab_train.yaml'
COLAB_CONFIG.parent.mkdir(exist_ok=True)
COLAB_CONFIG.write_text(yaml.safe_dump(settings, sort_keys=False), encoding='utf-8')

NEW_RUN = False
LEAKAGE_NOTE = 'Learning experiment; duplicate images exist across splits, evaluation is not independent'
print('Results and checkpoints:', PROJECT)
print('Config:', COLAB_CONFIG)


## ดูกราฟใน Notebook

เลือก run ใน TensorBoard เพื่อเปรียบเทียบผล การ resume เขียน log ต่อในโฟลเดอร์เดิม


In [ ]:
get_ipython().run_line_magic('load_ext', 'tensorboard')
get_ipython().run_line_magic('tensorboard', f'--logdir "{PROJECT}"')


## เริ่มหรือเทรนต่อ

Cell นี้เริ่มการฝึกจริง ระบบตรวจ dataset และ manifest ก่อนเริ่ม จะกลับไปใช้ epoch ล่าสุดที่อ่านได้จาก `resume.pt`, `resume_previous.pt`, `last.pt` หรือ `best.pt` ถ้ามีไฟล์ checkpoint แต่ไม่มีไฟล์ใดใช้ได้ จะหยุดพร้อมแจ้งสาเหตุโดยไม่ลบไฟล์หรือเริ่มใหม่เงียบ ๆ

หาก quota หมดจะต้องกลับมาเปิด GPU runtime และรัน Notebook ใหม่ด้วยตนเอง ไม่มีโค้ดต่อเวลา หลบ idle timeout หรือข้ามข้อจำกัด Colab


In [ ]:
command = [sys.executable, '-u', '-m', 'scripts.colab_train',
           '--config', str(COLAB_CONFIG), '--project', str(PROJECT),
           '--execute', '--leakage-note', LEAKAGE_NOTE]
if NEW_RUN:
    command.append('--new-run')
subprocess.run(command, check=True)


## ตำแหน่งไฟล์ที่เก็บใน Drive

แต่ละรอบอยู่ใน `MyDrive/PCB_YOLO11/experiments/pcb_<timestamp>/`

- `weights/best.pt` — โมเดลที่ดีที่สุดตาม validation สำหรับนำไปประเมิน/ใช้งาน
- `weights/last.pt` — checkpoint ล่าสุดของ Ultralytics
- `weights/resume.pt` และ `resume_previous.pt` — snapshot พร้อม optimizer สำหรับเทรนต่อ
- `progress.json` — epoch ที่บันทึกแล้ว หรือสถานะ completed
- `experiment.json`, `dataset_audit.json`, `results.csv` และ TensorBoard events — รายละเอียดการทดลอง

`best.pt` ยังต้องผ่านขั้นประเมินและ export metadata ก่อนใช้กับแอป PCB ตาม README ไม่เปลี่ยนชุดข้อมูลหรือขนาดโมเดลระหว่าง resume หากต้องการเปลี่ยนให้สร้างการทดลองใหม่
